# 평가(Eval) 만들기
어떤 작업에서 Claude가 최대한 높은 정확도를 내도록 최적화하는 일은 경험적 과학이자 지속적인 개선 과정입니다. 프롬프트를 바꿨더니 핵심 지표가 좋아졌는지 확인하려 할 때든, 모델이 프로덕션에 내보낼 만큼 충분히 좋은지 가늠하려 할 때든, 잘 만든 오프라인 평가 체계는 성공에 결정적입니다.

이 레시피에서는 평가를 구축할 때 자주 쓰이는 패턴과, 그 과정에서 참고할 만한 경험칙을 살펴봅니다.

## 평가의 구성 요소
평가는 보통 네 부분으로 이뤄집니다.
- 모델에 입력되는 입력 프롬프트. 이 프롬프트를 바탕으로 Claude가 답변을 생성하게 합니다. 평가를 설계할 때 입력 열에는 테스트 시점에 프롬프트 템플릿으로 주입되는 여러 가변 입력이 담기는 경우가 많습니다.
- 평가하려는 모델에 입력 프롬프트를 넣어 얻은 출력.
- 모델 출력과 비교할 "골든 답변". 골든 답변은 반드시 정확히 일치해야 하는 값일 수도 있고, 채점자가 점수를 매길 때 비교 기준으로 삼을 수 있는 이상적인 답변의 예시일 수도 있습니다.
- 아래에서 설명할 채점 방법 중 하나로 산출되는 점수. 해당 문항에서 모델이 얼마나 잘했는지를 나타냅니다.

## 평가 채점 방법
평가에서 시간과 비용이 많이 드는 부분은 두 가지입니다. 첫째는 평가 문항과 골든 답변을 작성하는 일이고, 둘째는 채점입니다. 이미 확보된 데이터셋이 없거나 문항을 수작업으로 만들지 않고 생성할 방법이 없다면 문항과 골든 답변 작성에 꽤 많은 시간이 들 수 있지만(문항 생성에 Claude를 활용해 보세요!), 대개 한 번만 들이면 되는 고정 비용이라는 장점이 있습니다. 문항과 골든 답변은 한 번 써 두면 다시 쓸 일이 거의 없습니다. 반면 채점은 평가를 다시 돌릴 때마다 계속 발생하는 비용이며, 평가는 앞으로도 여러 번 돌리게 될 가능성이 높습니다. 따라서 빠르고 저렴하게 채점할 수 있는 평가를 만드는 것이 설계의 중심이 되어야 합니다.

평가를 채점하는 흔한 방법은 세 가지입니다.
- **코드 기반 채점:** 일반적인 코드(주로 문자열 일치와 정규식)로 모델 출력을 채점하는 방식입니다. 정답과의 완전 일치를 확인하거나, 문자열에 특정 핵심 문구가 포함되어 있는지 확인하는 형태가 흔합니다. 이 방식이 가능하도록 평가를 설계할 수 있다면 단연 최선의 채점 방법입니다. 매우 빠르고 신뢰도가 높기 때문입니다. 다만 많은 평가가 이런 채점 방식에 맞지 않습니다.
- **사람 채점:** 사람이 모델이 생성한 답변을 보고 골든 답변과 비교해 점수를 매깁니다. 거의 모든 작업에 적용할 _수_ 있다는 점에서 가장 범용적인 채점 방법이지만, 그만큼 엄청나게 느리고 비쌉니다. 평가 규모가 크다면 더욱 그렇습니다. 가능하다면 사람 채점이 필요한 평가는 되도록 설계하지 않는 것이 좋습니다.
- **모델 기반 채점:** 알고 보면 Claude는 스스로를 채점하는 능력이 매우 뛰어나서, 창작 글의 어조 분석이나 자유 형식 질의응답의 정확도처럼 예전에는 사람이 해야 했던 다양한 작업의 채점에 활용할 수 있습니다. Claude용 *채점 프롬프트*를 작성해서 이를 수행합니다.

각 채점 방법의 예시를 하나씩 살펴보겠습니다.

### 코드 기반 채점
여기서는 어떤 대상의 다리가 몇 개인지 Claude가 정확히 맞히는지를 평가하고 채점해 보겠습니다. Claude가 다리 개수라는 숫자만 출력하기를 원하므로, 완전 일치 방식의 코드 기반 채점기를 쓸 수 있도록 평가를 설계합니다.

In [ ]:
# Install and read in required packages, plus create an anthropic client.
%pip install anthropic

In [2]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-opus-4-1"

In [6]:
# Define our input prompt template for the task.
def build_input_prompt(animal_statement):
    user_content = f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.

    Here is the animal statment.
    <animal_statement>{animal_statement}</animal_statment>

    How many legs does the animal have? Return just the number of legs as an integer and nothing else."""

    messages = [{"role": "user", "content": user_content}]
    return messages

In [4]:
# Define our eval (in practice you might do this as a jsonl or csv file instead).
eval = [
    {"animal_statement": "The animal is a human.", "golden_answer": "2"},
    {"animal_statement": "The animal is a snake.", "golden_answer": "0"},
    {
        "animal_statement": "The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.",
        "golden_answer": "5",
    },
]

In [7]:
# Get completions for each input.
# Define our get_completion function (including the stop sequence discussed above).
def get_completion(messages):
    response = client.messages.create(model=MODEL_NAME, max_tokens=5, messages=messages)
    return response.content[0].text


# Get completions for each question in the eval.
outputs = [get_completion(build_input_prompt(question["animal_statement"])) for question in eval]

# Let's take a quick look at our outputs
for output, question in zip(outputs, eval, strict=False):
    print(
        f"Animal Statement: {question['animal_statement']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n"
    )

Animal Statement: The animal is a human.
Golden Answer: 2
Output: 2

Animal Statement: The animal is a snake.
Golden Answer: 0
Output: 0

Animal Statement: The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.
Golden Answer: 5
Output: 5



In [8]:
# Check our completions against the golden answers.
# Define a grader function
def grade_completion(output, golden_answer):
    return output == golden_answer


# Run the grader function on our outputs and print the score.
grades = [
    grade_completion(output, question["golden_answer"])
    for output, question in zip(outputs, eval, strict=False)
]
print(f"Score: {sum(grades) / len(grades) * 100}%")

Score: 100.0%


### 사람 채점
이번에는 범용 챗봇 같은 용도로 Claude에 개방형 질문을 여러 개 던진 평가를 채점한다고 생각해 봅시다. 안타깝게도 답변이 제각각일 수 있어 코드로는 채점할 수 없습니다. 이럴 때 쓸 수 있는 방법 하나가 사람 채점입니다.

In [9]:
# Define our input prompt template for the task.
def build_input_prompt(question):
    user_content = f"""Please answer the following question:
    <question>{question}</question>"""

    messages = [{"role": "user", "content": user_content}]
    return messages

In [10]:
# Define our eval. For this task, the best "golden answer" to give a human are instructions on what to look for in the model's output.
eval = [
    {
        "question": "Please design me a workout for today that features at least 50 reps of pulling leg exercises, at least 50 reps of pulling arm exercises, and ten minutes of core.",
        "golden_answer": "A correct answer should include a workout plan with 50 or more reps of pulling leg exercises (such as deadlifts, but not such as squats which are a pushing exercise), 50 or more reps of pulling arm exercises (such as rows, but not such as presses which are a pushing exercise), and ten minutes of core workouts. It can but does not have to include stretching or a dynamic warmup, but it cannot include any other meaningful exercises.",
    },
    {
        "question": "Send Jane an email asking her to meet me in front of the office at 9am to leave for the retreat.",
        "golden_answer": "A correct answer should decline to send the email since the assistant has no capabilities to send emails. It is okay to suggest a draft of the email, but not to attempt to send the email, call a function that sends the email, or ask for clarifying questions related to sending the email (such as which email address to send it to).",
    },
    {
        "question": "Who won the super bowl in 2024 and who did they beat?",  # Claude should get this wrong since it comes after its training cutoff.
        "golden_answer": "A correct answer states that the Kansas City Chiefs defeated the San Francisco 49ers.",
    },
]

In [11]:
# Get completions for each input.
# Define our get_completion function (including the stop sequence discussed above).
def get_completion(messages):
    response = client.messages.create(model=MODEL_NAME, max_tokens=2048, messages=messages)
    return response.content[0].text


# Get completions for each question in the eval.
outputs = [get_completion(build_input_prompt(question["question"])) for question in eval]

# Let's take a quick look at our outputs
for output, question in zip(outputs, eval, strict=False):
    print(
        f"Question: {question['question']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n"
    )

Question: Please design me a workout for today that features at least 50 reps of pulling leg exercises, at least 50 reps of pulling arm exercises, and ten minutes of core.
Golden Answer: A correct answer should include a workout plan with 50 or more reps of pulling leg exercises (such as deadlifts, but not such as squats which are a pushing exercise), 50 or more reps of pulling arm exercises (such as rows, but not such as presses which are a pushing exercise), and ten minutes of core workouts. It can but does not have to include stretching or a dynamic warmup, but it cannot include any other meaningful exercises.
Output: Here's a workout plan for today that includes at least 50 reps of pulling leg exercises, 50 reps of pulling arm exercises, and ten minutes of core:

Pulling Leg Exercises:
1. Hamstring Curls (lying or seated): 3 sets of 12 reps (36 reps total)
2. Single-leg Romanian Deadlifts: 2 sets of 10 reps per leg (40 reps total)

Pulling Arm Exercises:
1. Bent-over Rows: 3 sets o

이 문항은 사람이 채점해야 하므로, 여기서부터는 직접 출력과 골든 답변을 비교해 평가하거나, 출력과 골든 답변을 CSV로 저장해 다른 채점자에게 넘기면 됩니다.

### 모델 기반 채점
위 평가를 매번 손으로 채점해야 한다면 금세 굉장히 번거로워집니다. 평가 규모가 현실적인 수준(수십, 수백, 심지어 수천 문항)이라면 더욱 그렇습니다. 다행히 더 나은 방법이 있습니다. 채점을 Claude에게 맡길 수 있습니다. 앞에서 쓴 평가와 응답을 그대로 사용해 어떻게 하는지 살펴보겠습니다.

In [12]:
# We start by defining a "grader prompt" template.
def build_grader_prompt(answer, rubric):
    user_content = f"""You will be provided an answer that an assistant gave to a question, and a rubric that instructs you on what makes the answer correct or incorrect.

    Here is the answer that the assistant gave to the question.
    <answer>{answer}</answer>

    Here is the rubric on what makes the answer correct or incorrect.
    <rubric>{rubric}</rubric>

    An answer is correct if it entirely meets the rubric criteria, and is otherwise incorrect. =
    First, think through whether the answer is correct or incorrect based on the rubric inside <thinking></thinking> tags. Then, output either 'correct' if the answer is correct or 'incorrect' if the answer is incorrect inside <correctness></correctness> tags."""

    messages = [{"role": "user", "content": user_content}]
    return messages


# Now we define the full grade_completion function.
import re


def grade_completion(output, golden_answer):
    messages = build_grader_prompt(output, golden_answer)
    completion = get_completion(messages)
    # Extract just the label from the completion (we don't care about the thinking)
    pattern = r"<correctness>(.*?)</correctness>"
    match = re.search(pattern, completion, re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        raise ValueError("Did not find <correctness></correctness> tags.")


# Run the grader function on our outputs and print the score.
grades = [
    grade_completion(output, question["golden_answer"])
    for output, question in zip(outputs, eval, strict=False)
]
print(f"Score: {grades.count('correct') / len(grades) * 100}%")

Score: 66.66666666666666%


보시다시피 Claude 기반 채점기는 Claude의 응답을 높은 정확도로 분석하고 채점해, 소중한 시간을 아껴 줍니다.

이제 평가를 위한 여러 채점 설계 패턴을 알게 되었으니, 직접 평가를 만들어 볼 준비가 되었습니다. 시작할 때 도움이 될 만한 몇 가지 조언을 남깁니다.
- 가능하면 평가를 여러분의 작업에 맞게 구체적으로 만들고, 평가 문항의 분포가 실제 질문과 난이도의 분포와 비슷해지도록 하세요.
- 모델 기반 채점기가 여러분의 작업을 잘 채점할 수 있는지 알아보는 유일한 방법은 직접 해 보는 것입니다. 한번 돌려 보고 표본을 몇 개 읽어 보면서 적합한 작업인지 확인하세요.
- 자동화 가능한 평가와 여러분 사이를 가로막는 것은 대개 영리한 설계뿐입니다. 작업의 본질을 해치지 않으면서도 채점을 자동화할 수 있는 형태로 문항을 구성해 보세요. 문항을 객관식으로 바꾸는 것이 흔히 쓰이는 방법입니다.
- 일반적으로 아주 적은 수의 고품질 문항보다는, 많은 수의 다소 낮은 품질 문항 쪽을 선호하는 것이 좋습니다.